In [ ]:
import numpy as np

# System and simulation parameters
d = 3        # state dimension
m = 1        # control dimension
T = 16       # horizon (number of time steps, note: controls are T-1)
N = 10000    # number of trajectories / PID simulations
dt = 0.1     # time step size

# Load the previously saved trajectories and extract initial states
trajectories = np.load("fractional_system_trajectories.npy")
initial_states = trajectories[:, 0, :]  # shape: (N, d)

# System matrices and nonlinear dynamics
A = np.array([[-0.1,  0.2,  0.0],
              [-0.2, -0.1,  0.3],
              [ 0.0, -0.3, -0.1]])
B = np.random.randn(d, m) * 0.1  # use a fixed B

def N_func(x):
    # x is a numpy array of shape (d,)
    return np.array([
        x[1]*x[2],
        -x[0]*x[2],
        x[0]*x[1]
    ])

# PID controller parameters
Kp = 1.0
Ki = 0.1
Kd = 0.05

c = np.array([1/3, 1/3, 1/3])

# Pre-allocate array to store PID control sequences.
pid_controls = np.zeros((N, T - 1, m))  # each control sequence has T-1 steps

# Loop over each initial state and simulate the system with PID control
for i in range(N):
    x = initial_states[i].copy()  # initial state, shape (d,)
    integrator = 0.0
    prev_error = 0.0
    
    for t in range(T - 1):
        # Compute the measured output (scalar)
        y = c.dot(x)
        error = 0 - y  # regulation: desired output is 0
        
        # Derivative term (finite difference)
        derivative = (error - prev_error) / dt
        
        # Integrator update
        integrator += error * dt
        
        # PID control law
        u = Kp * error + Ki * integrator + Kd * derivative
        
        # Store control signal (as a scalar in a 1-D array)
        pid_controls[i, t, 0] = u
        
        # Update the state using the nonlinear dynamics:
        x = x + dt * (A @ x + N_func(x) + B.flatten() * u)
        
        # Update previous error
        prev_error = error
        
    if i % 1000 == 0:
        print(f"Processed {i} / {N}")

# Save the PID control sequences
np.save("pid_controls.npy", pid_controls)